# ML-08 — Capstone Modeling Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

My lane is Refresh / Content Opportunity Scoring. The output is a ranked review queue, so I use predicted probabilities as ranking scores rather than treating the task as a simple yes/no decision.

I begin with Logistic Regression because it is reproducible, relatively interpretable, and provides probability scores. I also train a Random Forest as a nonlinear challenger. The Random Forest is retained only if it improves the primary metric, Precision@50, on the same held-out clients.

The proxy label remains `trend_direction == "down"` so the comparison is consistent with my earlier work. `trend_direction` and `trend_pct` are label-derived and are never model features.

The safe model features are:

- Log-transformed 90-day impressions
- Log-transformed 90-day sessions
- CTR
- Average position, treating zero as missing
- Content age
- Days since last update
- Days with impressions

In [1]:

from pathlib import Path
import subprocess
import pandas as pd
import numpy as np
import sklearn
from IPython.display import display

local_data = Path("data/raw/content_refresh_anonymized.csv")
colab_data = Path(
    "flyrank-starter/data/raw/content_refresh_anonymized.csv"
)

if local_data.exists():
    DATA_PATH = local_data
elif colab_data.exists():
    DATA_PATH = colab_data
else:
    subprocess.run(
        [
            "git",
            "clone",
            "--depth",
            "1",
            "https://github.com/flyrank-bih/"
            "flyrank-ml-internship-starter.git",
            "flyrank-starter",
        ],
        check=True,
    )
    DATA_PATH = colab_data

df = pd.read_csv(DATA_PATH)

# Proxy label, used only as y.
df["is_declining_proxy"] = (
    df["trend_direction"].eq("down").astype(int)
)

# Safe engineered features.
df["log_impressions_90d"] = np.log1p(
    df["impressions_90d"]
)

df["log_sessions_90d"] = np.log1p(
    df["sessions_90d"]
)

# In this dataset, zero means position data is unavailable.
df["avg_position_clean"] = (
    df["avg_position"].replace(0, np.nan)
)

FEATURE_COLS = [
    "log_impressions_90d",
    "log_sessions_90d",
    "ctr",
    "avg_position_clean",
    "content_age_days",
    "days_since_last_update",
    "days_with_impressions",
]

FORBIDDEN_FEATURES = {
    "trend_direction",
    "trend_pct",
    "is_declining_proxy",
    "content_id",
    "client_id",
}

assert set(FEATURE_COLS).isdisjoint(FORBIDDEN_FEATURES)

X = df[FEATURE_COLS].copy()
y = df["is_declining_proxy"].copy()
groups = df["client_id"].copy()

print("Dataset shape:", df.shape)
print("Content IDs unique:", df["content_id"].is_unique)
print("Clients:", groups.nunique())
print("Proxy-label base rate:", round(y.mean(), 3))
print("Scikit-learn version:", sklearn.__version__)

display(X.head())

Dataset shape: (30000, 48)
Content IDs unique: True
Clients: 32
Proxy-label base rate: 0.542
Scikit-learn version: 1.9.0


,log_impressions_90d,log_sessions_90d,ctr,avg_position_clean,content_age_days,days_since_last_update,days_with_impressions
0,8.243808,2.890372,0.76,10.6,187,20,88
1,9.636980,2.302585,0.05,20.3,445,25,88
2,9.440023,2.484907,0.09,36.5,141,20,88
3,9.371779,4.369448,0.49,6.2,463,22,88
4,9.859588,4.983607,0.13,44.0,263,14,88


## 2. Split design

I use a grouped client holdout with a fixed random seed. All pages from a client remain entirely in either training or testing, preventing the model from learning client-specific patterns and then being evaluated on other pages from the same client.

The starter dataset does not contain a sequence of independent historical snapshots suitable for a genuine past-to-future split, so grouped client validation is the most honest available design.

The Week-4 baseline rule is frozen without changing its thresholds or formula. It is recomputed on the same held-out test rows used for Logistic Regression and Random Forest. Every method is therefore compared using the same labels, test items and ranking metrics.

In [2]:
from sklearn.model_selection import GroupShuffleSplit

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.25,
    random_state=42,
)

train_index, test_index = next(
    splitter.split(X, y, groups=groups)
)

train_clients = set(groups.iloc[train_index])
test_clients = set(groups.iloc[test_index])

assert train_clients.isdisjoint(test_clients)

X_train = X.iloc[train_index]
X_test = X.iloc[test_index]

y_train = y.iloc[train_index]
y_test = y.iloc[test_index]

print("Training rows:", len(train_index))
print("Testing rows:", len(test_index))
print("Training clients:", len(train_clients))
print("Testing clients:", len(test_clients))
print(
    "Client overlap:",
    len(train_clients.intersection(test_clients)),
)
print(
    "Training base rate:",
    round(y_train.mean(), 3),
)
print(
    "Testing base rate:",
    round(y_test.mean(), 3),
)

Training rows: 22885
Testing rows: 7115
Training clients: 24
Testing clients: 8
Client overlap: 0
Training base rate: 0.55
Testing base rate: 0.517


## 3. Train + compare vs my baseline

The frozen Week-4 baseline prioritizes pages with at least 500 impressions and at least 91 days since their recorded update. Its score remains:

`log(1 + impressions) × staleness weight`

Logistic Regression and Random Forest are trained only on the training clients. Missing numeric values are imputed using medians learned from the training split. Logistic Regression additionally standardizes its inputs.

The primary metric is Precision@50 because the practical decision is which 50 pages a reviewer should open first. Precision@10 and Precision@20 show performance at smaller review capacities. Average Precision and ROC AUC provide supporting whole-ranking measurements.

In [3]:
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
)

# Readable primary model.
logistic_model = make_pipeline(
    SimpleImputer(
        strategy="median",
        add_indicator=True,
    ),
    StandardScaler(),
    LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        random_state=42,
    ),
)

# Nonlinear challenger.
forest_model = make_pipeline(
    SimpleImputer(
        strategy="median",
        add_indicator=True,
    ),
    RandomForestClassifier(
        n_estimators=300,
        min_samples_leaf=10,
        max_features="sqrt",
        class_weight="balanced_subsample",
        random_state=42,
        n_jobs=-1,
    ),
)

logistic_model.fit(X_train, y_train)
forest_model.fit(X_train, y_train)

logistic_scores = logistic_model.predict_proba(
    X_test
)[:, 1]

forest_scores = forest_model.predict_proba(
    X_test
)[:, 1]


# ------------------------------------------------------------
# Freeze and recompute the exact Week-4 baseline on the same
# test rows. No label field enters this score.
# ------------------------------------------------------------

test_rows = df.iloc[test_index].copy()

staleness_weight = 1 + (
    test_rows["days_since_last_update"]
    .sub(90)
    .clip(lower=0, upper=180)
    / 180
)

baseline_eligible = (
    (test_rows["impressions_90d"] >= 500)
    & (test_rows["days_since_last_update"] >= 91)
)

baseline_scores = np.where(
    baseline_eligible,
    np.log1p(test_rows["impressions_90d"])
    * staleness_weight,
    0.0,
)


def precision_at_k(scores, labels, k):
    scores = np.asarray(scores)
    labels = np.asarray(labels)

    ranking = np.argsort(
        -scores,
        kind="stable",
    )

    selected = ranking[: min(k, len(ranking))]
    return float(labels[selected].mean())


def evaluate_ranking(name, scores):
    return {
        "method": name,
        "precision_at_10": precision_at_k(
            scores, y_test, 10
        ),
        "precision_at_20": precision_at_k(
            scores, y_test, 20
        ),
        "precision_at_50": precision_at_k(
            scores, y_test, 50
        ),
        "average_precision": average_precision_score(
            y_test, scores
        ),
        "roc_auc": roc_auc_score(
            y_test, scores
        ),
    }


test_base_rate = float(y_test.mean())

comparison_rows = [
    {
        "method": "Naive base rate",
        "precision_at_10": test_base_rate,
        "precision_at_20": test_base_rate,
        "precision_at_50": test_base_rate,
        "average_precision": test_base_rate,
        "roc_auc": 0.5,
    },
    evaluate_ranking(
        "Week-4 rule baseline",
        baseline_scores,
    ),
    evaluate_ranking(
        "Logistic Regression",
        logistic_scores,
    ),
    evaluate_ranking(
        "Random Forest",
        forest_scores,
    ),
]

comparison = (
    pd.DataFrame(comparison_rows)
    .sort_values(
        "precision_at_50",
        ascending=False,
    )
    .reset_index(drop=True)
)

display(comparison.round(3))


# Select the learned model using the declared primary metric.
learned_models = {
    "Logistic Regression": (
        logistic_model,
        logistic_scores,
    ),
    "Random Forest": (
        forest_model,
        forest_scores,
    ),
}

comparison_by_method = comparison.set_index("method")

selected_name = max(
    learned_models,
    key=lambda name: comparison_by_method.loc[
        name, "precision_at_50"
    ],
)

selected_model, selected_scores = learned_models[
    selected_name
]

print("Selected learned model:", selected_name)
print(
    "Selection criterion: highest held-out "
    "Precision@50 among learned models."
)

,method,precision_at_10,precision_at_20,precision_at_50,average_precision,roc_auc
0,Logistic Regression,0.900,0.850,0.840,0.618,0.608
1,Random Forest,0.800,0.650,0.640,0.626,0.627
2,Naive base rate,0.517,0.517,0.517,0.517,0.500
3,Week-4 rule baseline,0.300,0.200,0.300,0.508,0.489


Selected learned model: Logistic Regression
Selection criterion: highest held-out Precision@50 among learned models.


## 4. Errors and interpretation

I interpret feature importance using permutation importance on the held-out clients. This measures how much Average Precision decreases when one feature is shuffled; it describes model reliance, not causation.

I also inspect the three highest-confidence classification errors using a probability threshold of 0.5. Classification errors are secondary to Precision@50, but they help reveal sparse or ambiguous cases.

The proxy label describes an observed current-window decline bucket. Therefore, even a correct prediction does not prove that refreshing the page would cause traffic to recover.

In [4]:
from sklearn.inspection import permutation_importance

permutation = permutation_importance(
    selected_model,
    X_test,
    y_test,
    n_repeats=5,
    random_state=42,
    scoring="average_precision",
    n_jobs=-1,
)

importance_table = (
    pd.DataFrame({
        "feature": FEATURE_COLS,
        "importance_mean": permutation.importances_mean,
        "importance_std": permutation.importances_std,
    })
    .sort_values(
        "importance_mean",
        ascending=False,
    )
    .reset_index(drop=True)
)

print("Permutation importance on held-out clients")
display(importance_table.round(4))


# ------------------------------------------------------------
# Error analysis
# ------------------------------------------------------------

predicted_labels = (
    selected_scores >= 0.5
).astype(int)

error_table = test_rows[
    ["content_id", *FEATURE_COLS]
].copy()

error_table["actual_label"] = y_test.to_numpy()
error_table["predicted_label"] = predicted_labels
error_table["predicted_probability"] = selected_scores

false_positives = (
    (error_table["actual_label"] == 0)
    & (error_table["predicted_label"] == 1)
).sum()

false_negatives = (
    (error_table["actual_label"] == 1)
    & (error_table["predicted_label"] == 0)
).sum()

error_table["error_confidence"] = np.where(
    error_table["actual_label"] == 0,
    error_table["predicted_probability"],
    1 - error_table["predicted_probability"],
)

wrong_cases = (
    error_table.loc[
        error_table["actual_label"]
        != error_table["predicted_label"]
    ]
    .sort_values(
        "error_confidence",
        ascending=False,
    )
    .head(3)
    .copy()
)


def explain_error(row):
    if (
        row["actual_label"] == 1
        and row["days_with_impressions"] <= 5
    ):
        return (
            "False negative with very sparse search evidence; "
            "the model has little stable history to use."
        )

    if row["actual_label"] == 1:
        return (
            "False negative: the page declined despite signals "
            "that resembled non-declining pages."
        )

    return (
        "False positive: its age or visibility resembled a "
        "declining page, but the observed proxy stayed non-down."
    )


wrong_cases["why_this_is_hard"] = wrong_cases.apply(
    explain_error,
    axis=1,
)

display_columns = [
    "content_id",
    "actual_label",
    "predicted_label",
    "predicted_probability",
    "log_impressions_90d",
    "log_sessions_90d",
    "ctr",
    "avg_position_clean",
    "content_age_days",
    "days_since_last_update",
    "days_with_impressions",
    "why_this_is_hard",
]

pd.set_option("display.max_colwidth", None)

print("Three highest-confidence wrong cases")
display(wrong_cases[display_columns])


# ------------------------------------------------------------
# Short interpretation
# ------------------------------------------------------------

selected_p50 = comparison_by_method.loc[
    selected_name,
    "precision_at_50",
]

baseline_p50 = comparison_by_method.loc[
    "Week-4 rule baseline",
    "precision_at_50",
]

top_features = importance_table.head(3)[
    "feature"
].tolist()

print(
    f"{selected_name} achieved held-out Precision@50 "
    f"of {selected_p50:.3f}, compared with "
    f"{baseline_p50:.3f} for the frozen Week-4 rule."
)

print(
    "The three strongest permutation signals were:",
    top_features,
)

print(
    f"At the 0.5 threshold there were "
    f"{false_positives} false positives and "
    f"{false_negatives} false negatives."
)

print(
    "The reviewed errors show that sparse histories and "
    "pages whose age/visibility patterns resemble another "
    "class remain difficult."
)

Permutation importance on held-out clients


,feature,importance_mean,importance_std
0,content_age_days,0.0532,0.0042
1,log_sessions_90d,0.0444,0.0013
2,days_with_impressions,0.0361,0.0020
3,avg_position_clean,0.0217,0.0018
4,log_impressions_90d,0.0068,0.0010
5,ctr,0.0016,0.0006
6,days_since_last_update,-0.0001,0.0016


Three highest-confidence wrong cases


,content_id,actual_label,predicted_label,predicted_probability,log_impressions_90d,log_sessions_90d,ctr,avg_position_clean,content_age_days,days_since_last_update,days_with_impressions,why_this_is_hard
27271,content_7bc32bc1df59,1,0,0.008529,0.693147,0.693147,0.0,NaN,238,92,1,False negative with very sparse search evidence; the model has little stable history to use.
24849,content_2f002563e9cd,1,0,0.145473,2.890372,3.637586,0.0,5.2,502,20,11,False negative: the page declined despite signals that resembled non-declining pages.
17690,content_c268b1716236,1,0,0.162133,1.386294,1.098612,0.0,41.7,502,20,2,False negative with very sparse search evidence; the model has little stable history to use.


Logistic Regression achieved held-out Precision@50 of 0.840, compared with 0.300 for the frozen Week-4 rule.
The three strongest permutation signals were: ['content_age_days', 'log_sessions_90d', 'days_with_impressions']
At the 0.5 threshold there were 1604 false positives and 1376 false negatives.
The reviewed errors show that sparse histories and pages whose age/visibility patterns resemble another class remain difficult.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.